# Soft Margins & Tuning C

**Companion lesson:** https://ml-viz.vercel.app/courses/svm/03-soft-margins

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Hinge-loss SVM by gradient descent

The soft-margin SVM is just hinge loss + L2, so we can train it with plain subgradient descent and watch C work.

In [ ]:
def make_data(n=80, overlap=1.2):
    X0 = np.random.randn(n // 2, 2) * overlap + [-1.5, -1]
    X1 = np.random.randn(n // 2, 2) * overlap + [1.5, 1]
    X = np.vstack([X0, X1]); y = np.array([-1] * (n // 2) + [1] * (n // 2))
    return X, y

X, y = make_data()

def train_svm(X, y, C, epochs=400, lr=0.01):
    w = np.zeros(2); b = 0.0
    for _ in range(epochs):
        margins = y * (X @ w + b)
        viol = margins < 1                     # only violators have gradient
        w -= lr * (w - C * (y[viol, None] * X[viol]).sum(axis=0))
        b -= lr * (-C * y[viol].sum())
    return w, b

## The same data at three prices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
for ax, C in zip(axes, [0.01, 1.0, 100.0]):
    w, b = train_svm(X, y, C)
    Z = (xx * w[0] + yy * w[1] + b)
    ax.contourf(xx, yy, np.sign(Z), levels=1, colors=['#6366f133', '#14b8a633'])
    for lv, ls in [(-1, ':'), (0, '-'), (1, ':')]:
        ax.contour(xx, yy, Z, levels=[lv], colors='white', linestyles=ls, linewidths=1.2)
    ax.scatter(*X[y == -1].T, c='#6366f1', s=18); ax.scatter(*X[y == 1].T, c='#14b8a6', s=18)
    margin = 2 / np.linalg.norm(w)
    ax.set_title(f'C = {C}   margin = {margin:.2f}')
plt.tight_layout(); plt.show()
# small C: wide margin, many violations tolerated
# large C: narrow margin, boundary contorts to satisfy stragglers

## Count the violators

In [ ]:
for C in [0.01, 1.0, 100.0]:
    w, b = train_svm(X, y, C)
    xi = np.maximum(0, 1 - y * (X @ w + b))
    print(f'C={C:>6}:  margin={2/np.linalg.norm(w):5.2f}   '
          f'violators(ξ>0)={int((xi > 1e-9).sum()):2d}   misclassified(ξ>1)={int((xi > 1).sum()):2d}')

**Try it:** add one extreme outlier with `X = np.vstack([X, [[-4, 3]]]); y = np.append(y, 1)` and retrain at each C. Watch large C reshape the whole boundary for one point while small C shrugs.